# Music Annotation Project V8.4 — Notebook 41
## Learned Structurally Valuable PRINT Selection

**Runtime: GPU.** Development-only, source-group OOF challenger.

This notebook forms connected candidate groups from frozen V8.2 pixels with `0.50 <= p_print < 0.75`, then learns two separate outcomes: **genuine-PRINT safety** and **counterfactual structural value**. Musical structure names and continuation rules are never inference inputs. Clean truth is loaded only after the candidate manifest is hashed. V7.8 remains unchanged except where a selected group copies exact observed bytes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-learn torch torchvision matplotlib joblib

## 1. Configuration and pre-registration

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8';os.environ['PYTHONHASHSEED']='41001'
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,random,shutil,sys,platform,gc,zipfile
import cv2,torch,joblib,numpy as np,pandas as pd,matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import GroupKFold,LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression,Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score,roc_auc_score,brier_score_loss,mean_absolute_error
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject');V82_ROOT=ROOT/'observed_print_classifier_v8_2';OUTROOT=ROOT/'observed_print_classifier_v8_4'
SOURCE_V82_RUN_OVERRIDE=None;SEED=41001;LOWER_P=.50;UPPER_P=.75;MIN_GROUP_PIXELS=2;LOCAL=33;WIDE=97;WH=WIDE//2
BATCH=128;EPOCHS=14;LR=8e-4;WD=2e-4;BOOT=10000;VALUE_RADIUS=12
SAFETY_T=.995;VALUE_T=.90;MAX_PAGE_FRACTION=.10;MAX_COMPONENT_FRACTION=.20
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge','p_print'];ARMS=['probability_only','local_33','multiscale_33_97']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu';torch.use_deterministic_algorithms(True)
if DEVICE=='cuda':torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_4_notebook41';OUT=OUTROOT/RUN;INPUTS=OUT/'inputs';TABLES=OUT/'tables';MODELS=OUT/'models';WHOLE=OUT/'whole_pages';AUDIT=OUT/'audit';VISUAL=OUT/'visual_audit';UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MODELS,WHOLE,AUDIT,VISUAL,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
PRE={'version':'8.4-notebook41-development','development_only':True,'promotion_authority':False,'probability_interval':[LOWER_P,UPPER_P],'grouping':{'connectivity':8,'minimum_pixels':MIN_GROUP_PIXELS},'patches':[LOCAL,WIDE],'channels':CHANNELS,'arms':ARMS,'safety_threshold':SAFETY_T,'value_threshold':VALUE_T,'semantic_labels_as_inputs':False,'seed':SEED}
(OUT/'01_pre_registration.json').write_text(json.dumps(PRE,indent=2));print('Device',DEVICE,'Output',OUT)

## 2. Resolve frozen V8.2 development evidence

In [ ]:
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def oh(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def ah(a):
 a=np.ascontiguousarray(a);return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()
def latest(root,req):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
SOURCE=Path(SOURCE_V82_RUN_OVERRIDE) if SOURCE_V82_RUN_OVERRIDE else latest(V82_ROOT,['experiment_identity.json','tables/all_eligible_oof_predictions.parquet','whole_pages'])
if SOURCE is None:raise RuntimeError('No complete V8.2 run')
ident=json.loads((SOURCE/'experiment_identity.json').read_text())
if ident.get('version')!='8.2-development-whole-page' or ident.get('independent_data_used') is not False:raise RuntimeError('Wrong V8.2 source role')
V75=Path(ident['v7_5_reference']);pred_path=SOURCE/'tables/all_eligible_oof_predictions.parquet';full=pd.read_parquet(pred_path).reset_index(drop=True)
req={'page','source_group','y','x','truth','fold','raw_print','p_print'}
if req-set(full):raise RuntimeError(f'Missing columns {req-set(full)}')
for c in ['y','x','truth','fold']:full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
for c in ['raw_print','p_print']:full[c]=pd.to_numeric(full[c],errors='raise').astype(float)
if full.duplicated(['page','y','x']).any() or not np.isfinite(full[['raw_print','p_print']]).all().all():raise RuntimeError('Prediction integrity failure')
record={'source':str(SOURCE),'identity_sha256':sha(SOURCE/'experiment_identity.json'),'predictions_sha256':sha(pred_path),'v75':str(V75)};(OUT/'source_record.json').write_text(json.dumps(record,indent=2))

## 3. Build and lock lower-probability connected candidate groups

In [ ]:
status=pd.read_csv(V75/'tables/cache_status.csv');cache={str(r.page):Path(r.cache) for _,r in status.iterrows()};ctx={};rows=[];pix=[]
arrs=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page,g in full.groupby('page',sort=True):
 with np.load(cache[page]) as z:
  miss=[k for k in arrs if k not in z.files]
  if miss:raise RuntimeError(f'{page} missing {miss}')
  a={k:z[k].copy() for k in arrs}
 obs=cv2.imread(str(SOURCE/'whole_pages'/page/'observed.png'),0);base=cv2.imread(str(SOURCE/'whole_pages'/page/'v7_8.png'),0);mask=a['mask']>0
 if obs is None or base is None or obs.shape!=base.shape or not np.array_equal(base[~mask],obs[~mask]):raise RuntimeError(f'{page} baseline failure')
 paper=np.clip(cv2.GaussianBlur(obs,(0,0),45),1,255).astype(np.uint8);norm=cv2.divide(obs,paper,scale=255);depth=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5);edge=cv2.Canny(obs,70,170)>0
 prob=np.zeros(obs.shape,np.float32);eligible=np.zeros(obs.shape,bool);ys=g.y.to_numpy(int);xs=g.x.to_numpy(int);prob[ys,xs]=g.p_print.to_numpy();eligible[ys,xs]=True
 band=eligible&(prob>=LOWER_P)&(prob<UPPER_P);n,lab,st,_=cv2.connectedComponentsWithStats(band.astype(np.uint8),8);_,mcomp,mst,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
 d={'obs':obs,'baseline':base,'mask':mask,'norm':norm,'depth':depth,'edge':edge,'p_print':prob,'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,'candidate_labels':lab,'mask_component':mcomp};ctx[page]=d
 for cid in range(1,n):
  area=int(st[cid,cv2.CC_STAT_AREA])
  if area<MIN_GROUP_PIXELS:continue
  yy,xx=np.where(lab==cid);pr=prob[yy,xx];mc=mcomp[yy,xx];primary=int(pd.Series(mc[mc>0]).mode().iloc[0]) if np.any(mc>0) else 0;gid=f'{page}:g{cid:05d}'
  rows.append({'group_id':gid,'page':page,'source_group':str(g.source_group.iloc[0]),'fold':int(g.fold.iloc[0]),'candidate_group_id':cid,'mask_component_id':primary,'pixels':area,'centre_y':float(yy.mean()),'centre_x':float(xx.mean()),'p_mean':float(pr.mean()),'p_min':float(pr.min()),'p_max':float(pr.max()),'p_std':float(pr.std()),'mask_depth_mean':float(depth[yy,xx].mean()),'edge_rate':float(edge[yy,xx].mean()),'horizontal_rate':float(d['horizontal'][yy,xx].mean()),'vertical_rate':float(d['vertical'][yy,xx].mean()),'compact_rate':float(d['compact'][yy,xx].mean()),'ambiguous_rate':float(d['ambiguous'][yy,xx].mean())})
  pix.extend({'group_id':gid,'page':page,'y':int(y),'x':int(x)} for y,x in zip(yy,xx))
cand=pd.DataFrame(rows).sort_values(['source_group','page','candidate_group_id']).reset_index(drop=True);pixels=pd.DataFrame(pix).sort_values(['group_id','y','x'])
if cand.empty:raise RuntimeError('No registered candidate groups')
cand.to_parquet(TABLES/'02_candidate_group_manifest.parquet',index=False);pixels.to_parquet(TABLES/'02_candidate_group_pixels.parquet',index=False)
lock={'manifest':sha(TABLES/'02_candidate_group_manifest.parquet'),'pixels':sha(TABLES/'02_candidate_group_pixels.parquet'),'source':sha(pred_path),'constants':PRE};(OUT/'candidate_lock.json').write_text(json.dumps({'lock_id':oh(lock),'payload':lock},indent=2));print('Groups',len(cand),'pixels',len(pixels))

## 4. Construct post-lock safety and counterfactual value labels

In [ ]:
if not (OUT/'candidate_lock.json').exists():raise RuntimeError('Candidate lock missing')
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
def ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def f1(ref,pred,reg,tol=1):
 ref=ref&reg;pred=pred&reg
 if not ref.any():return np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8);p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1);r=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum();return float(2*p*r/max(p+r,1e-12))
labels=[]
for page,gc in cand.groupby('page'):
 clean=cv2.imread(str(clean_map[page]),0);d=ctx[page];ci=ink(clean);ce=cv2.Canny(clean,70,170)>0;bi=ink(d['baseline']);be=cv2.Canny(d['baseline'],70,170)>0;lookup=full[full.page==page].set_index(['y','x'])
 for _,r in gc.iterrows():
  gm=d['candidate_labels']==int(r.candidate_group_id);yy,xx=np.where(gm);truth=np.array([int(lookup.loc[(int(y),int(x)),'truth']) for y,x in zip(yy,xx)]);tu=int((truth==2).sum());tp=int((truth==1).sum());fa=int((truth==0).sum());safe_eligible=tu==0;safe=int(safe_eligible and fa==0 and tp>0)
  out=d['baseline'].copy();out[gm]=d['obs'][gm];reg=cv2.dilate(gm.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25)))>0;base_mae=float(np.abs(clean[reg].astype(float)-d['baseline'][reg]).mean());new_mae=float(np.abs(clean[reg].astype(float)-out[reg]).mean());mae_gain=base_mae-new_mae
  ig=(f1(ci,ink(out),reg)-f1(ci,bi,reg)) if np.isfinite(f1(ci,bi,reg)) else 0.;eg=(f1(ce,cv2.Canny(out,70,170)>0,reg)-f1(ce,be,reg)) if np.isfinite(f1(ce,be,reg)) else 0.;value=float(mae_gain+8*ig+8*eg)
  labels.append({'group_id':r.group_id,'truth_print_pixels':tp,'truth_annotation_pixels':fa,'truth_uncertain_pixels':tu,'safety_label_eligible':safe_eligible,'safety_label':safe,'mae_benefit':mae_gain,'ink_f1_gain':ig,'edge_f1_gain':eg,'value_score':value,'value_positive':int(value>0)})
data=cand.merge(pd.DataFrame(labels),on='group_id',validate='one_to_one');data.to_parquet(TABLES/'05_counterfactual_value_labels.parquet',index=False);print(data[['safety_label','value_positive','value_score']].describe())

## 5. Observable-only scalar, 33×33 and 33×33 + 97×97 models

In [ ]:
padded={}
for page,d in ctx.items():
 padded[page]={}
 for k in CHANNELS:
  a=d[k].astype(np.float32);a=a/255 if k in ['obs','norm'] else np.clip(a/25,0,1) if k=='depth' else a;padded[page][k]=np.pad(a,WH,mode='reflect')
def patch(page,cy,cx,size):
 cy=int(round(cy))+WH;cx=int(round(cx))+WH;h=size//2;return np.stack([padded[page][k][cy-h:cy+h+1,cx-h:cx+h+1] for k in CHANNELS]).astype(np.float32)
FEATURES=['pixels','p_mean','p_min','p_max','p_std','mask_depth_mean','edge_rate','horizontal_rate','vertical_rate','compact_rate','ambiguous_rate'];XS=data[FEATURES].astype(float)
class DS(Dataset):
 def __init__(self,idx):self.idx=np.asarray(idx,int)
 def __len__(self):return len(self.idx)
 def __getitem__(self,i):
  j=int(self.idx[i]);r=data.iloc[j];return torch.from_numpy(patch(r.page,r.centre_y,r.centre_x,LOCAL)),torch.from_numpy(patch(r.page,r.centre_y,r.centre_x,WIDE)),torch.tensor(float(r.safety_label)),torch.tensor(float(r.value_positive)),torch.tensor(float(r.value_score)),torch.tensor(j)
def loader(idx,shuffle,seed):
 g=torch.Generator();g.manual_seed(seed);return DataLoader(DS(idx),batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,generator=g)
class Enc(nn.Module):
 def __init__(self):super().__init__();self.n=nn.Sequential(nn.Conv2d(len(CHANNELS),24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
 def forward(self,x):return self.n(x).flatten(1)
class MT(nn.Module):
 def __init__(self,multi):super().__init__();self.multi=multi;self.a=Enc();self.b=Enc() if multi else None;self.s=nn.Sequential(nn.Linear(128 if multi else 64,64),nn.ReLU(),nn.Dropout(.15));self.hs=nn.Linear(64,1);self.hv=nn.Linear(64,1);self.hr=nn.Linear(64,1)
 def forward(self,a,b):
  z=self.a(a);z=torch.cat([z,self.b(b)],1) if self.multi else z;z=self.s(z);return self.hs(z).squeeze(1),self.hv(z).squeeze(1),self.hr(z).squeeze(1)
def seed(s):random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s) if torch.cuda.is_available() else None
def fit(idx,multi,f):
 seed(SEED+f);m=MT(multi).to(DEVICE);opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WD);y1=data.iloc[idx].safety_label.to_numpy();y2=data.iloc[idx].value_positive.to_numpy();l1=nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(y1)-y1.sum())/max(y1.sum(),1),device=DEVICE));l2=nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(y2)-y2.sum())/max(y2.sum(),1),device=DEVICE));lr=nn.SmoothL1Loss();hist=[]
 for ep in range(EPOCHS):
  m.train();tot=n=0
  for a,b,ys,yv,vr,_ in loader(idx,True,SEED+ep):
   a,b,ys,yv,vr=a.to(DEVICE),b.to(DEVICE),ys.to(DEVICE),yv.to(DEVICE),vr.to(DEVICE);opt.zero_grad();ps,pv,pr=m(a,b);loss=l1(ps,ys)+l2(pv,yv)+.2*lr(pr,vr);loss.backward();opt.step();tot+=float(loss.detach())*len(ys);n+=len(ys)
  hist.append(tot/n)
 return m,hist
def predict(m,idx):
 out=[];m.eval()
 with torch.no_grad():
  for a,b,_,_,_,j in loader(idx,False,SEED):
   ps,pv,pr=m(a.to(DEVICE),b.to(DEVICE));out.extend((int(k),float(x),float(y),float(z)) for k,x,y,z in zip(j,torch.sigmoid(ps).cpu(),torch.sigmoid(pv).cpu(),pr.cpu()))
 return out

## 6. Source-group OOF training and cross-fitted calibration

In [ ]:
groups=data.source_group.astype(str);splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(10,groups.nunique()));folds=list(splitter.split(data,groups=groups));pd.DataFrame([{'fold':i,'test_groups':'|'.join(sorted(groups.iloc[te].unique())),'test_rows':len(te)} for i,(tr,te) in enumerate(folds,1)]).to_csv(TABLES/'03_source_group_fold_map.csv',index=False)
oof=[];inventory=[]
for f,(tr,te) in enumerate(folds,1):
 if set(groups.iloc[tr])&set(groups.iloc[te]):raise RuntimeError('Source leakage')
 st=np.array([i for i in tr if bool(data.iloc[i].safety_label_eligible)])
 if len(st)<20 or data.iloc[st].safety_label.nunique()<2:raise RuntimeError(f'Fold {f} safety variation insufficient')
 sm=Pipeline([('i',SimpleImputer()),('s',StandardScaler()),('m',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]);vm=Pipeline([('i',SimpleImputer()),('s',StandardScaler()),('m',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]);vr=Pipeline([('i',SimpleImputer()),('s',StandardScaler()),('m',Ridge(alpha=2))]);sm.fit(XS.iloc[st],data.iloc[st].safety_label);vm.fit(XS.iloc[tr],data.iloc[tr].value_positive);vr.fit(XS.iloc[tr],data.iloc[tr].value_score)
 oof.extend({'row_id':int(j),'fold':f,'arm':'probability_only','raw_safety':float(a),'raw_value':float(b),'pred_value':float(c)} for j,a,b,c in zip(te,sm.predict_proba(XS.iloc[te])[:,1],vm.predict_proba(XS.iloc[te])[:,1],vr.predict(XS.iloc[te])))
 for arm,multi in [('local_33',False),('multiscale_33_97',True)]:
  m,h=fit(tr,multi,f);p=MODELS/f'{arm}_fold{f}.pt';torch.save(m.state_dict(),p);inventory.append({'arm':arm,'fold':f,'path':str(p),'sha256':sha(p),'final_loss':h[-1]});oof.extend({'row_id':j,'fold':f,'arm':arm,'raw_safety':a,'raw_value':b,'pred_value':c} for j,a,b,c in predict(m,te));del m;gc.collect()
oof=pd.DataFrame(oof);pd.DataFrame(inventory).to_csv(AUDIT/'model_inventory.csv',index=False)
cal=[]
for arm,q in oof.groupby('arm'):
 q=q.sort_values('row_id').reset_index(drop=True);ps=np.full(len(q),np.nan);pv=np.full(len(q),np.nan)
 for f in sorted(q.fold.unique()):
  tr=q.fold!=f;te=q.fold==f;safe=tr&data.loc[q.row_id,'safety_label_eligible'].to_numpy(bool);is1=IsotonicRegression(out_of_bounds='clip').fit(q.loc[safe,'raw_safety'],data.loc[q.loc[safe,'row_id'],'safety_label']);is2=IsotonicRegression(out_of_bounds='clip').fit(q.loc[tr,'raw_value'],data.loc[q.loc[tr,'row_id'],'value_positive']);ps[te]=is1.predict(q.loc[te,'raw_safety']);pv[te]=is2.predict(q.loc[te,'raw_value'])
 q['p_safety']=ps;q['p_value']=pv;cal.append(q)
oof=pd.concat(cal).merge(data.reset_index().rename(columns={'index':'row_id'}),on='row_id',validate='many_to_one');oof['eligible_select']=(oof.p_safety>=SAFETY_T)&(oof.p_value>=VALUE_T)&(oof.pred_value>0)&oof.safety_label_eligible;oof['selected']=False;oof['reason']='below_rule'
for arm,qa in oof.groupby('arm'):
 for page,qp in qa.groupby('page'):
  c=qp[qp.eligible_select].sort_values(['p_safety','p_value','pred_value'],ascending=False);cap=max(1,int(MAX_PAGE_FRACTION*len(qp)));chosen=[];cc={}
  for idx,r in c.iterrows():
   comp=int(r.mask_component_id);limit=max(1,int(MAX_COMPONENT_FRACTION*(qp.mask_component_id==comp).sum()))
   if len(chosen)>=cap:break
   if cc.get(comp,0)>=limit:continue
   chosen.append(idx);cc[comp]=cc.get(comp,0)+1
  oof.loc[chosen,['selected','reason']]=[True,'selected'];oof.loc[c.index.difference(chosen),'reason']='concentration_cap'
oof.to_parquet(TABLES/'04_oof_safety_and_value_predictions.parquet',index=False)

## 7. Evaluate arms, assemble pages and apply development gates

In [ ]:
def upper(e,n,z=1.96):
 if n==0:return np.nan
 p=e/n;d=1+z*z/n;c=p+z*z/(2*n);r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return (c+r)/d
def summarise(q):
 s=q.selected;page=q.groupby('page').apply(lambda g:pd.Series({'selected':int(g.selected.sum()),'tp':int(g.loc[g.selected,'truth_print_pixels'].sum()),'fp':int(g.loc[g.selected,'truth_annotation_pixels'].sum()),'value':float(g.loc[g.selected,'value_score'].sum())}),include_groups=False).reset_index();rng=np.random.default_rng(SEED);idx=page.set_index('page');names=page.page.to_numpy();boot=[]
 for _ in range(BOOT):
  b=idx.loc[rng.choice(names,len(names),replace=True)];boot.append(float(b.value.sum()))
 e=int((s&(q.safety_label==0)).sum());n=int(s.sum())
 return {'arm':q.arm.iloc[0],'groups':len(q),'selected_groups':n,'false_safe_groups':e,'true_print_pixels':int(q.loc[s,'truth_print_pixels'].sum()),'false_annotation_pixels':int(q.loc[s,'truth_annotation_pixels'].sum()),'realised_value':float(q.loc[s,'value_score'].sum()),'value_95_low':float(np.quantile(boot,.025)),'false_group_wilson95_upper':upper(e,n),'selected_pages':int((page.selected>0).sum())}
summary=pd.DataFrame([summarise(q) for _,q in oof.groupby('arm')]);summary.to_csv(TABLES/'arm_comparison_summary.csv',index=False);pages=[]
for arm,qa in oof.groupby('arm'):
 for page,qp in qa.groupby('page'):
  d=ctx[page];sm=np.zeros(d['mask'].shape,bool)
  for _,r in qp[qp.selected].iterrows():sm|=d['candidate_labels']==int(r.candidate_group_id)
  out=d['baseline'].copy();out[sm]=d['obs'][sm];outside=int((out[~d['mask']]!=d['obs'][~d['mask']]).sum());reg=int((out[~sm]!=d['baseline'][~sm]).sum());wrong=int(((out!=d['baseline'])&(out!=d['obs'])).sum())
  if outside or reg or wrong:raise RuntimeError(f'{arm}/{page} integrity')
  p=WHOLE/arm/page;p.mkdir(parents=True,exist_ok=False);cv2.imwrite(str(p/'observed.png'),d['obs']);cv2.imwrite(str(p/'v7_8.png'),d['baseline']);cv2.imwrite(str(p/'v8_4.png'),out);dm=np.zeros_like(out);dm[sm]=255;cv2.imwrite(str(p/'selected_groups.png'),dm);pages.append({'arm':arm,'page':page,'source_group':qp.source_group.iloc[0],'candidate_groups':len(qp),'selected_groups':int(qp.selected.sum()),'selected_pixels':int(sm.sum()),'outside_mask_drift':outside,'nonselected_regression':reg,'wrong_changed_byte':wrong})
pdf=pd.DataFrame(pages);pdf.to_csv(AUDIT/'page_output_integrity.csv',index=False);by_page=oof.groupby(['arm','page','source_group'],as_index=False).apply(lambda g:pd.Series({'candidate_groups':len(g),'selected_groups':int(g.selected.sum()),'false_safe_groups':int((g.selected&(g.safety_label==0)).sum()),'true_print_pixels':int(g.loc[g.selected,'truth_print_pixels'].sum()),'false_annotation_pixels':int(g.loc[g.selected,'truth_annotation_pixels'].sum()),'value_score':float(g.loc[g.selected,'value_score'].sum())}),include_groups=False);by_page.to_csv(TABLES/'06_results_by_page.csv',index=False);by_source=by_page.groupby(['arm','source_group'],as_index=False).sum(numeric_only=True);by_source.to_csv(TABLES/'07_results_by_source_group.csv',index=False)
review=oof[(oof.selected)|((~oof.selected)&(oof.safety_label==1)&(oof.value_score>0))].copy();review['priority']=np.select([review.selected&(review.safety_label==0),review.selected&(review.value_score<=0),review.selected],['false_safe','negative_value','selected_benefit'],default='missed_benefit');review.to_csv(TABLES/'11_visual_review_queue.csv',index=False)
s=summary.set_index('arm');m=s.loc['multiscale_33_97'];l=s.loc['local_33'];p=s.loc['probability_only'];src=by_source[by_source.arm=='multiscale_33_97'];G={'V4_1_integrity':len(oof)==len(data)*3,'V4_2_immutability':pdf[['outside_mask_drift','nonselected_regression','wrong_changed_byte']].to_numpy().sum()==0,'V4_3_zero_false_safe':m.false_safe_groups==0,'V4_4_false_bound':np.isfinite(m.false_group_wilson95_upper) and m.false_group_wilson95_upper<=.05,'V4_5_positive_value':m.realised_value>0 and m.value_95_low>0,'V4_6_multiscale_dominance':m.realised_value>l.realised_value and m.realised_value>p.realised_value,'V4_7_coverage':m.selected_groups>=30 and m.selected_pages>=10 and src.loc[src.selected_groups>0,'source_group'].nunique()>=3,'V4_8_source_stability':(src.false_safe_groups==0).all() and (src.value_score>=0).all(),'V4_9_semantic_breadth':'POST_HOC_REVIEW_REQUIRED','V4_10_visual_audit':'QUEUE_GENERATED_NOT_COMPLETED'};first=next((k for k,v in G.items() if v is False),None);verdict={'status':'PASS_PENDING_REVIEW' if first is None else 'FAIL','first_failed_gate':first,'gates':G,'development_only':True};(OUT/'15_v8_4_verdict.json').write_text(json.dumps(verdict,indent=2));print(summary.to_string(index=False));print(json.dumps(verdict,indent=2))

## 8. Manifest and standard final upload package

In [ ]:
env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()};manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.4-notebook41-development','pre_registration':PRE,'source_record':record,'candidate_lock':json.loads((OUT/'candidate_lock.json').read_text()),'groups':len(data),'pages':data.page.nunique(),'source_groups':data.source_group.nunique(),'models':inventory,'verdict':verdict,'environment':env};(OUT/'14_experiment_manifest.json').write_text(json.dumps(manifest,indent=2,default=str));lines=['V8.4 NOTEBOOK 41','',f'Run: {OUT}',f'Candidate groups: {len(data)}',f'Pages: {data.page.nunique()} | source groups: {data.source_group.nunique()}',f'Verdict: {verdict["status"]}',f'First failed gate: {verdict["first_failed_gate"]}','',summary.to_string(index=False),'','No musical structure or continuation label was used as an inference input. This is development evidence only.'];(OUT/'16_executive_summary.txt').write_text(chr(10).join(lines))
files=[OUT/'01_pre_registration.json',OUT/'candidate_lock.json',OUT/'source_record.json',OUT/'14_experiment_manifest.json',OUT/'15_v8_4_verdict.json',OUT/'16_executive_summary.txt',TABLES/'02_candidate_group_manifest.parquet',TABLES/'05_counterfactual_value_labels.parquet',TABLES/'04_oof_safety_and_value_predictions.parquet',TABLES/'arm_comparison_summary.csv',TABLES/'06_results_by_page.csv',TABLES/'07_results_by_source_group.csv',TABLES/'11_visual_review_queue.csv',AUDIT/'model_inventory.csv',AUDIT/'page_output_integrity.csv'];missing=[str(p) for p in files if not p.exists()]
if missing:raise RuntimeError(f'Upload missing {missing}')
inv=[]
for p in files:
 rel=p.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);inv.append({'relative_path':str(rel),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'README.txt').write_text('V8.4 Notebook 41 development bundle. Start with 16_executive_summary.txt and 15_v8_4_verdict.json.');(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inv));zp=UPLOAD/'notebook41_v8_4_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file() and p!=zp:z.write(p,p.relative_to(UPLOAD))
print(chr(10).join(lines));print('Upload',zp)

## Review order
1. `16_executive_summary.txt`
2. `15_v8_4_verdict.json`
3. `tables/arm_comparison_summary.csv`
4. `tables/11_visual_review_queue.csv`
5. `whole_pages/multiscale_33_97/<page>/v7_8.png` versus `v8_4.png`
6. `upload/notebook41_v8_4_upload_bundle.zip`

Do not relax thresholds or concentration controls after reading the results. A material change is a new pre-registered challenger.